# RSNA knee: side classifier + ROI boxes, inference

Three stages, each a module of `src/infer/` run as its own process (as in `biohub-inf`), writing to `OUT_DIR`:

1. **Preprocess** (`infer.prep`): DICOM series to 32 x 128 x 128 volumes through exactly the training operations (checked voxel-identical on training series), plus geometry, slice order and the Laterality tag. CPU-bound, parallel over series.
2. **Side** (`infer.side`): left / right for *every* series, since a study can hold both knees. The model is used, not the DICOM tag: the tag is wrong on ~2% of tagged series. Writes `series_laterality.csv`.
3. **ROIs** (`infer.seg`): sagittal and coronal series flipped to a right knee by their own side, ROI U-Net, exactly one box per class (ACL, MCL, menisci, the three OA compartments, Baker's), each with a confidence. Failed boxes are replaced by registration from the same knee's other series. Writes `rois.csv`.

## Imports

In [1]:
import os
import sys
import glob
import json
import time
import subprocess

import numpy as np
import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # Code and wheels come from attached datasets; the competition image lacks
    # dynamic_network_architectures (the ROI U-Net) and the DICOM decoders.
    !cp -r /kaggle/input/datasets/theoviel/rsna-knee-code/src ./
    !pip install --quiet --no-index --find-links /kaggle/input/datasets/theoviel/rsna-knee-wheels dynamic-network-architectures pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm
    SRC = os.path.abspath("./src")
else:
    SRC = os.path.abspath("../src")

# Every stage runs as `<this python> -m infer.<stage>` with src/ on the path (sys.executable:
# a bare `python` may not be the kernel's interpreter).
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, SRC)
print("src:", SRC, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| CPUs:", os.cpu_count())

src: /data/rsna_knee/src | GPU: NVIDIA H100 80GB HBM3 | CPUs: 208


## Parameters

In [2]:
if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
    SIDE_W = "/kaggle/input/datasets/theoviel/rsna-knee-weights/side_2026-10-09_0/"
    ROI_W = "/kaggle/input/datasets/theoviel/rsna-knee-weights/roi_2026-10-09_1/"
    OUT_DIR = "/tmp/knee/"
else:
    DATA_DIR = "../input/"
    SIDE_W = "../logs/side_run/2026-10-09/0/"
    ROI_W = "../logs/roi_run/2026-10-09/1/"
    OUT_DIR = "../output/inference/"

CSV_NAME = "test_series.csv"       # series -> study, plane
SERIES_DIR = "test_series"         # <SERIES_DIR>/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
WORKERS = os.cpu_count()           # DICOM decoding is the bottleneck: one process per core
LIMIT = None                       # first N studies only (debugging)

for w in (SIDE_W, ROI_W):
    assert os.path.exists(os.path.join(w, "ckpt_final.pt")), f"missing weights: {w}"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_DIR, SIDE_W, ROI_W, OUT_DIR = map(os.path.abspath, (DATA_DIR, SIDE_W, ROI_W, OUT_DIR))

## 1. Preprocess the DICOMs

Decode, order the slices, normalise and resample every series; volumes go to `OUT_DIR/volumes.npy`, their metadata to `series.csv`, the file behind each slice index to `instances.csv` and the geometry to `geometry.json`.

In [3]:
t0 = time.time()
cmd = (f"{sys.executable} -m infer.prep --data_dir {DATA_DIR} --out_dir {OUT_DIR} --workers {WORKERS}"
       f" --csv_name {CSV_NAME} --series_dir {SERIES_DIR}" + (f" --limit {LIMIT}" if LIMIT else ""))
print(cmd)
!{cmd}
series = pd.read_csv(os.path.join(OUT_DIR, "series.csv"))
print(f"{time.time() - t0:.0f}s |", series.plane.value_counts().to_dict(), "| failed:", int((series.status != "ok").sum()))

/data/rsna_knee/.venv/bin/python3 -m infer.prep --data_dir /data/rsna_knee/input --out_dir /data/rsna_knee/output/inference --workers 208 --csv_name test_series.csv --series_dir test_series


-> 15/15 series of 3 studies in 4s (0.30 s/series), 0 flipped to the training left-right convention -> /data/rsna_knee/output/inference


5s | {'Sagittal': 7, 'Axial': 4, 'Coronal': 4} | failed: 0


## 2. Side classifier

One prediction per series (`p_right`, `side`); `dicom_laterality` is kept for reference only.

In [4]:
t0 = time.time()
!{sys.executable} -m infer.side --out_dir {OUT_DIR} --weights {SIDE_W}
laterality = pd.read_csv(os.path.join(OUT_DIR, "series_laterality.csv"))
laterality.to_csv("series_laterality.csv", index=False)
print(f"{time.time() - t0:.0f}s")
laterality.head()

-> 15 series: {'R': 10, 'L': 5}; 0 studies with both knees; agreement with the DICOM tag where present 1.000 (5 series)


4s


,SeriesInstanceUID,StudyInstanceUID,plane,p_right,side,dicom_laterality
0,1.2.826.0.1.3680043.8.498.11580656442259111255...,1.2.826.0.1.3680043.8.498.10047035057544427318...,Axial,0.028436,L,L
1,1.2.826.0.1.3680043.8.498.17811502614030631664...,1.2.826.0.1.3680043.8.498.10047035057544427318...,Sagittal,0.001988,L,L
2,1.2.826.0.1.3680043.8.498.30565395595045942404...,1.2.826.0.1.3680043.8.498.10047035057544427318...,Sagittal,0.001455,L,L
3,1.2.826.0.1.3680043.8.498.32856494541816845805...,1.2.826.0.1.3680043.8.498.10047035057544427318...,Coronal,0.028436,L,L
4,1.2.826.0.1.3680043.8.498.44334485654554877495...,1.2.826.0.1.3680043.8.498.10047035057544427318...,Axial,0.038466,L,L


## 3. ROI boxes

One box per (sagittal / coronal series, class), in the series' own voxel indices (slices in `instances.csv` order), with `confidence` = U-Net sharpness x support from the knee's other series, and `source` = `unet` or `registered`.

In [5]:
t0 = time.time()
!{sys.executable} -m infer.seg --out_dir {OUT_DIR} --weights {ROI_W}
rois = pd.read_csv(os.path.join(OUT_DIR, "rois.csv"))
rois.to_csv("rois.csv", index=False)
print(f"{time.time() - t0:.0f}s | {len(rois)} boxes, {rois.SeriesInstanceUID.nunique()} series")
rois.groupby("class")[["confidence", "support"]].mean().round(3)

-> 88 ROIs for 11 series (11 sagittal / coronal) in 3s (U-Net 2s); 0 below the 0.5 threshold, 0 of them replaced by registration; mean confidence 0.974


6s | 88 boxes, 11 series


,confidence,support
class,,
ACL,0.973,1.0
Baker's,0.969,1.0
Lateral Meniscus,0.977,1.0
Lateral OA,0.979,1.0
MCL,0.969,1.0
Medial Meniscus,0.970,1.0
Medial OA,0.977,1.0
PF OA,0.977,1.0
